# Audience Comment Cleaning Pipeline
**GATES Manfluencer Project — Nigeria Audience Reception**

Loads each audience comment dataset, cleans it, and saves the cleaned version back in place.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 100)

ROOT = Path.cwd()
COMMENTS_DIR = ROOT / "Nigeria/Audience Comments - Raw"

# Creator metadata
CREATOR_META = {
    "Banky Wellington":  {"orientation": "progressive", "platform": "YouTube"},
    "Deyemi Okanlawon": {"orientation": "progressive", "platform": "X"},
    "Wizarab":           {"orientation": "regressive",  "platform": "X"},
    "Shola":             {"orientation": "regressive",  "platform": "X"},
    "Agba John Doe":     {"orientation": "regressive",  "platform": "X"},
}

# Dataset registry: (creator, source_post) → file path
DATASETS = [
    ("Banky Wellington", "Final Say Faith",
     COMMENTS_DIR / "Banky Wellington" / "Final Say Faith.xlsx"),
    ("Banky Wellington", "My Story Journey Through Hope and Faith",
     COMMENTS_DIR / "Banky Wellington" / "My Story Journey Through Hope and Faith.xlsx"),
    ("Shola", "Women and Availability Trap",
     COMMENTS_DIR / "Shola" / "Women and Availability Trap.xlsx"),
    ("Shola", "7 Women Will Beg One Man to Marry",
     COMMENTS_DIR / "Shola" / "7 Women Will Beg One Man to Marry.xlsx"),
    ("Wizarab", "Sex Toys and Raping Young Boys",
     COMMENTS_DIR / "Wizarab" / "Sex Toys and Raping Young Boys.xlsx"),
    ("Wizarab", "Child Support and Divorce",
     COMMENTS_DIR / "Wizarab" / "Child Support and Divorce.xlsx"),
    ("Deyemi Okanlawon", "Men Must Hold Men Accountable",
     COMMENTS_DIR / "Deyemi Okanlawon" / "Men Must Hold Men Accountable.xlsx"),
    ("Deyemi Okanlawon", "Stop Raping Women Response",
     COMMENTS_DIR / "Deyemi Okanlawon" / "Stop Raping Women Response.xlsx"),
    ("Agba John Doe", "Never Leave Marriage Because Husband Cheated",
     COMMENTS_DIR / "Agba John Doe" / "Never Leave Marriage Because Husband Cheated.xlsx"),
    ("Agba John Doe", "Single Woman Earning Above 1.5M",
     COMMENTS_DIR / "Agba John Doe" / "Single Woman Earning Above 1.5M.xlsx"),
]

MIN_TEXT_LENGTH = 5

print(f"Datasets registered: {len(DATASETS)}")
for creator, post, path in DATASETS:
    print(f"  [{'OK' if path.exists() else 'MISSING'}] {creator} / {post}")

## 2. Cleaning Functions

In [ ]:
def normalize_text(s) -> str:
    """Unicode-normalize and collapse whitespace."""
    if not isinstance(s, str):
        s = str(s) if pd.notna(s) else ""
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("\u2018", "'").replace("\u2019", "'")
    s = s.replace("\u201c", '"').replace("\u201d", '"')
    s = re.sub(r"\s+", " ", s).strip()
    return s


def strip_tz(ts_series):
    """Parse timestamps and strip timezone for Excel compatibility."""
    ts = pd.to_datetime(ts_series, format="mixed", utc=True, errors="coerce")
    return ts.dt.tz_localize(None)


def extract_author(url_series):
    """Extract @handle from X URLs."""
    return url_series.apply(
        lambda u: u.split("x.com/")[1].split("/")[0]
        if pd.notna(u) and "x.com/" in str(u) else ""
    )


def clean_file(df, platform):
    """Auto-detect schema (raw or already-cleaned) and produce clean output."""
    cols = set(df.columns)
    
    # ── YouTube ──
    if platform == "YouTube":
        if "voteCount" in cols:
            out = pd.DataFrame({
                "author": df["author"],
                "comment": df["comment"].apply(normalize_text),
                "likes": pd.to_numeric(df["voteCount"], errors="coerce").fillna(0).astype(int),
                "reply_count": pd.to_numeric(df["replyCount"], errors="coerce").fillna(0).astype(int),
            })
        else:
            out = df.copy()
            out["comment"] = out["comment"].apply(normalize_text)
        text_col = "comment"
    
    # ── X / Twitter ──
    else:
        if "createdAt" in cols:
            out = pd.DataFrame({
                "author": extract_author(df["url"]),
                "text": df["text"].apply(normalize_text),
                "likes": pd.to_numeric(df.get("likeCount", 0), errors="coerce").fillna(0).astype(int),
                "replies": pd.to_numeric(df.get("replyCount", 0), errors="coerce").fillna(0).astype(int),
                "retweets": pd.to_numeric(df.get("retweetCount", 0), errors="coerce").fillna(0).astype(int),
                "timestamp": strip_tz(df["createdAt"]),
                "url": df["url"],
            })
        elif "id" in cols and "images" in cols:
            out = pd.DataFrame({
                "author": extract_author(df["url"]),
                "text": df["text"].apply(normalize_text),
                "likes": pd.to_numeric(df["likes"], errors="coerce").fillna(0).astype(int),
                "replies": pd.to_numeric(df["replies"], errors="coerce").fillna(0).astype(int),
                "retweets": pd.to_numeric(df["retweets"], errors="coerce").fillna(0).astype(int),
                "timestamp": strip_tz(df["timestamp"]),
                "url": df["url"],
            })
        else:
            out = df.copy()
            out["text"] = out["text"].apply(normalize_text)
            if "timestamp" in out.columns and hasattr(out["timestamp"].dtype, "tz") and out["timestamp"].dtype.tz:
                out["timestamp"] = out["timestamp"].dt.tz_localize(None)
        text_col = "text"
    
    # ── Drop junk rows ──
    before = len(out)
    out = out[out[text_col].ne("") & out[text_col].ne("nan")].copy()
    out = out[out[text_col].str.len() >= MIN_TEXT_LENGTH].copy()
    out = out.drop_duplicates(subset=[text_col]).copy()
    out = out.reset_index(drop=True)
    removed = before - len(out)
    
    return out, before, removed

print("Cleaning functions ready.")

## 3. Clean & Save Each Dataset

In [ ]:
results = []

for creator, source_post, path in DATASETS:
    platform = CREATOR_META[creator]["platform"]
    
    df_raw = pd.read_excel(path)
    df_clean, before, removed = clean_file(df_raw, platform)
    df_clean.to_excel(path, index=False)
    
    results.append({
        "creator": creator,
        "source_post": source_post,
        "platform": platform,
        "raw_rows": before,
        "cleaned_rows": len(df_clean),
        "removed": removed,
        "columns": list(df_clean.columns),
    })
    
    print(f"  {creator:20s} / {source_post:45s}  {before:>5} → {len(df_clean):>5}  (dropped {removed})")

print(f"\nDone. All 10 datasets cleaned and saved in place.")

## 4. Verify

In [ ]:
report = pd.DataFrame(results)
report["pct_kept"] = (report["cleaned_rows"] / report["raw_rows"] * 100).round(1)
print(report[["creator", "source_post", "platform", "raw_rows", "cleaned_rows", "removed", "pct_kept"]].to_string(index=False))
print(f"\nTotal: {report['raw_rows'].sum():,} raw → {report['cleaned_rows'].sum():,} cleaned ({report['removed'].sum():,} removed)")

In [ ]:
# Quick peek at each saved file to confirm
for creator, source_post, path in DATASETS:
    df = pd.read_excel(path)
    print(f"\n{creator} / {source_post}")
    print(f"  Shape: {df.shape}  |  Columns: {list(df.columns)}")
    print(f"  Sample: {df.iloc[0, df.columns.get_loc(df.columns[df.columns.str.contains('text|comment', case=False)][0])][:100]}")